# Phase 02: Scaled Extraction — Typhoon OCR Batch Pipeline

**Plan:** 02-01 — Scaled Extraction Mechanism & Rate Limiting

This notebook runs Typhoon OCR across **all 250+ PDFs** for Uthai Thani Constituency 2,
strictly throttling API calls to respect Typhoon's rate limits:
- Max **20 requests per minute**
- Max **2 requests per second**
- Enforced via `time.sleep(3)` between calls (≈ 1 req / 3 s → 20/min)

**Engine:** Typhoon OCR exclusively (D-01). No PaddleOCR or Tesseract.

**Error policy (D-02):** Failures are caught, logged, and skipped — the pipeline never crashes.
Failed paths are written to `logs/failed_extractions.json` for re-run.

**Target form:** ส.ส.5/18 (election day voting results per polling station)

## Cell 1: Environment Imports

In [1]:
import os
import re
import io
import json
import time
import tempfile
import logging
import concurrent.futures
import threading

import cv2
import numpy as np
from pathlib import Path
from datetime import datetime, UTC

import fitz  # PyMuPDF
from PIL import Image
from tqdm.notebook import tqdm
from typhoon_ocr import ocr_document

# Load .env for API keys
try:
    from dotenv import load_dotenv
    load_dotenv()
    print('dotenv loaded')
except ImportError:
    print('python-dotenv not installed; skipping .env load')

print(f'PyMuPDF version: {fitz.version}')
print('All imports OK')

dotenv loaded
PyMuPDF version: ('1.27.2.2', '1.27.2', None)
All imports OK


## Cell 2: Configuration

In [2]:
# ---------------------------------------------------------------------------
# Path configuration
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path().resolve()
SOURCE_DIR = Path(
    os.getenv(
        "SOURCE_DIR",
        str(NOTEBOOK_DIR.parent / "source" / "\u0e40\u0e02\u0e15\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u0e48 2")
    )
)
LOGS_DIR = NOTEBOOK_DIR.parent / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Rate-limit settings (D-03)
# Typhoon API limits: 20 req/min and 2 req/sec
# Strategy: sleep 3 seconds between calls => ~20 req/min with safety margin
# ---------------------------------------------------------------------------
INTER_REQUEST_DELAY = 3.0   # seconds between Typhoon API calls
MAX_REQUESTS_PER_MIN = 20   # documented limit (for logging only)

# ---------------------------------------------------------------------------
# OCR settings
# ---------------------------------------------------------------------------
PDF_DPI = 200
TARGET_FORM = "\u0e2a.\u0e2a.5\u0e17\u0e31\u0e1a18.pdf"  # ส.ส.5ทับ18.pdf

# ---------------------------------------------------------------------------
# Typhoon OCR API key
# ---------------------------------------------------------------------------
TYPHOON_API_KEY = os.getenv("TYPHOON_OCR_API_KEY")
TYPHOON_BASE_URL = os.getenv("TYPHOON_OCR_BASE_URL", "https://api.openthai.ai/v1")

# Validate environment
env_path = NOTEBOOK_DIR.parent / ".env"
if not env_path.exists():
    print(f"WARNING: .env not found at {env_path}")
    print("Copy .env.example to .env and set TYPHOON_OCR_API_KEY")

print(f"SOURCE_DIR:        {SOURCE_DIR}")
print(f"SOURCE_DIR exists: {SOURCE_DIR.exists()}")
print(f"LOGS_DIR:          {LOGS_DIR}")
print(f"Typhoon API key set: {bool(TYPHOON_API_KEY)}")
print(f"Typhoon base URL:    {TYPHOON_BASE_URL}")
print(f"Rate limit strategy: sleep {INTER_REQUEST_DELAY}s between calls")
print(f"Target form: {TARGET_FORM}")

SOURCE_DIR:        /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2
SOURCE_DIR exists: True
LOGS_DIR:          /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/logs
Typhoon API key set: True
Typhoon base URL:    https://api.opentyphoon.ai/v1
Rate limit strategy: sleep 3.0s between calls
Target form: ส.ส.5ทับ18.pdf


## Cell 3: PDF Discovery

In [3]:
def find_election_day_pdfs(root_dir: Path, form_pattern: str = '5ทับ18') -> list[dict]:
    """
    Recursively locate all ส.ส.5/18 election-day PDF forms.
    Matches filenames containing `form_pattern` but excludes advance-voting
    variants that contain 'บช' (บัญชีรายชื่อ / out-of-district ballots).
    """
    results = []
    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if (form_pattern in filename
                    and filename.endswith('.pdf')
                    and 'บช' not in filename):
                p = Path(dirpath)
                parts = p.relative_to(root_dir).parts
                results.append({
                    'path':        str(p / filename),
                    'filename':    filename,
                    'district':    parts[0] if len(parts) > 0 else '',
                    'subdistrict': parts[1] if len(parts) > 1 else '',
                    'station':     parts[2] if len(parts) > 2 else p.name,
                })
    results.sort(key=lambda x: x['path'])
    return results


all_pdfs = find_election_day_pdfs(SOURCE_DIR)
print(f"Found {len(all_pdfs)} election-day PDFs (บช variants excluded)")
if all_pdfs:
    print(f"First: {all_pdfs[0]['path']}")
    print(f"Last:  {all_pdfs[-1]['path']}")

from collections import Counter
variant_counts = Counter(e['filename'] for e in all_pdfs)
print(f"\nFilename variants ({len(variant_counts)} unique):")
for name, count in sorted(variant_counts.items(), key=lambda x: -x[1]):
    print(f"  {count:4d}  {name}")


Found 133 election-day PDFs (บช variants excluded)
First: /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2/อำเภอบ้านไร่/ตำบลทัพหลวง/หน่วยเลือกตั้งที่ 1/ส.ส.5ทับ18 1.pdf
Last:  /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2/อำเภอลานสัก/ตำบลระบำ/หน่วยเลือกตั้งที่ 11/5ทับ18 11 ระบำ.pdf

Filename variants (73 unique):
    60  ส.ส.5ทับ18.pdf
     2  ส.ส.5ทับ18 4.pdf
     1  ส.ส.5ทับ18 1.pdf
     1  ส.ส.5ทับ18 10.pdf
     1  ส.ส.5ทับ18  11.pdf
     1  ส.ส.5ทับ18 12.pdf
     1  ส.ส.5ทับ18  13.pdf
     1  ส.ส.5ทับ18  14.pdf
     1  ส.ส.5ทับ18  15.pdf
     1  ส.ส.5ทับ18 2.pdf
     1  ส.ส.5ทับ18 3.pdf
     1  ส.ส.5ทับ18 5.pdf
     1  ส.ส. 5ทับ18  7.pdf
     1  ส.ส.5ทับ18 8.pdf
     1  ส.ส.5ทับ18 9.pdf
     1  ส.ส. 5ทับ18 หนองจอก-1.pdf
     1  ส.ส. 5ทับ18 หนองจอก-2.pdf
     1  ส.ส. 5ทับ18 หนองจอก-19.pdf
     1  ส.ส. 5ทับ18 หนองจอก-20.pdf
     1  ส.ส. 5ทับ18 หนองจอก-21.pdf
     1  ส.ส. 5ทับ18 หนองจอ

## Cell 4: Typhoon OCR Helper (Full-Page Extraction)

In [4]:
OCR_TIMEOUT = 20  # seconds — abandon call and log as failure if exceeded


def score_page_grid(img_pil) -> float:
    """
    Score a PIL image for table/grid density using OpenCV morphological operations.
    Detects horizontal and vertical line structures characteristic of vote-count tables.
    Returns sum of detected line pixels — higher = more grid content.
    """
    gray = np.array(img_pil.convert('L'))
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Horizontal lines: wide kernel
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    h_lines  = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, h_kernel)

    # Vertical lines: tall kernel
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
    v_lines  = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, v_kernel)

    return float(np.sum(h_lines) + np.sum(v_lines))


def select_best_page_jpeg(pdf_path: str, dpi: int = 200) -> tuple[str, int, float]:
    """
    Render every page of the PDF, score each for grid density, and save the
    highest-scoring page as a temp JPEG.

    Always returns a result — no absolute threshold. The highest relative score
    within the document wins, so no PDF is ever skipped.

    Returns (jpeg_path, best_page_num_0indexed, best_score).
    Caller must delete the temp file.
    """
    doc = fitz.open(pdf_path)
    best_img   = None
    best_score = -1.0
    best_page  = 0

    for page_num in range(len(doc)):
        page = doc[page_num]
        mat  = fitz.Matrix(dpi / 72, dpi / 72)
        pix  = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
        img  = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
        score = score_page_grid(img)
        if score > best_score:
            best_score = score
            best_img   = img
            best_page  = page_num

    doc.close()

    tmp = tempfile.NamedTemporaryFile(suffix='.jpg', delete=False)
    best_img.save(tmp.name, 'JPEG', quality=95)
    tmp.close()
    return tmp.name, best_page, best_score


def run_typhoon_ocr_full_page(pdf_path: str) -> str:
    """
    Select the best page via OpenCV grid scoring, then run Typhoon OCR with
    a hard timeout using a daemon thread.
    """
    jpeg_path, best_page, score = select_best_page_jpeg(pdf_path)

    result_box = [None]
    error_box  = [None]

    def _call():
        try:
            result_box[0] = ocr_document(
                pdf_or_image_path=jpeg_path,
                figure_language='Thai',
            )
        except Exception as exc:
            error_box[0] = exc

    t = threading.Thread(target=_call, daemon=True)
    t.start()
    t.join(timeout=OCR_TIMEOUT)

    try:
        os.unlink(jpeg_path)
    except FileNotFoundError:
        pass

    if t.is_alive():
        raise TimeoutError(f"Typhoon OCR timed out after {OCR_TIMEOUT}s (page {best_page}, score {score:.0f})")
    if error_box[0] is not None:
        raise error_box[0]
    return result_box[0], best_page, score


## Cell 5: Vote Parsing (Regex)

In [5]:
THAI_DIGIT_MAP = str.maketrans(
    '\u0e50\u0e51\u0e52\u0e53\u0e54\u0e55\u0e56\u0e57\u0e58\u0e59',
    '0123456789'
)


def thai_to_arabic(text: str) -> str:
    """Convert Thai numerals to Arabic numerals."""
    return text.translate(THAI_DIGIT_MAP)


def parse_vote_counts(ocr_text: str) -> dict:
    """
    Parse numerical vote fields from full-page ส.ส.5/18 OCR text.

    Extracts:
      - eligible_voters   : ผู้มีสิทธิเลือกตั้ง
      - ballots_issued    : บัตรที่ออกให้
      - valid_ballots     : บัตรดี
      - spoiled_ballots   : บัตรเสีย
      - no_vote_ballots   : ไม่ประสงค์จะลงคะแนน
      - total_votes       : รวมคะแนน
      - candidate_votes   : {candidate_number: vote_count} dict
    """
    text = thai_to_arabic(ocr_text)
    result = {
        'eligible_voters': None,
        'ballots_issued': None,
        'valid_ballots': None,
        'spoiled_ballots': None,
        'no_vote_ballots': None,
        'total_votes': None,
        'candidate_votes': {},
    }

    # Eligible voters: ผู้มีสิทธิเลือกตั้ง ... <number>
    m = re.search(
        r'\u0e1c\u0e39\u0e49\u0e21\u0e35\u0e2a\u0e34\u0e17\u0e18\u0e34'
        r'\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07[^\d]*(\d[\d,]*)',
        text
    )
    if m:
        result['eligible_voters'] = int(m.group(1).replace(',', ''))

    # Valid ballots: บัตรดี
    m = re.search(
        r'\u0e1a\u0e31\u0e15\u0e23\u0e14\u0e35[^\d]*(\d[\d,]*)',
        text
    )
    if m:
        result['valid_ballots'] = int(m.group(1).replace(',', ''))

    # Spoiled ballots: บัตรเสีย
    m = re.search(
        r'\u0e1a\u0e31\u0e15\u0e23\u0e40\u0e2a\u0e35\u0e22[^\d]*(\d[\d,]*)',
        text
    )
    if m:
        result['spoiled_ballots'] = int(m.group(1).replace(',', ''))

    # No-vote ballots: ไม่ประสงค์จะลงคะแนน
    m = re.search(
        r'\u0e44\u0e21\u0e48\u0e1b\u0e23\u0e30\u0e2a\u0e07\u0e04\u0e4c[^\d]*(\d[\d,]*)',
        text
    )
    if m:
        result['no_vote_ballots'] = int(m.group(1).replace(',', ''))

    # Total votes: รวมคะแนน
    m = re.search(
        r'\u0e23\u0e27\u0e21\u0e04\u0e30\u0e41\u0e19\u0e19[^\d]*(\d[\d,]*)',
        text
    )
    if m:
        result['total_votes'] = int(m.group(1).replace(',', ''))

    # Candidate votes: หมายเลข N ... <votes>
    # Pattern: candidate number followed by vote count on same or adjacent line
    candidate_matches = re.findall(
        r'\u0e2b\u0e21\u0e32\u0e22\u0e40\u0e25\u0e02\s*(\d+)[^\d]*(\d[\d,]*)',
        text
    )
    for cand_num, votes in candidate_matches:
        result['candidate_votes'][int(cand_num)] = int(votes.replace(',', ''))

    return result


print('parse_vote_counts() defined.')

parse_vote_counts() defined.


## Cell 8: Subset Verification (5 PDFs)

Run a quick dry-run on 5 PDFs to verify rate throttling is visible.
Run this cell BEFORE the full batch to confirm the pipeline works.

Expected: tqdm progress bar shows ~3 seconds per iteration.

In [ ]:
VERIFY_SUBSET_SIZE = 5
subset_pdfs = all_pdfs[:VERIFY_SUBSET_SIZE]

print(f"Running subset verification on {len(subset_pdfs)} PDFs")
print(f"Expected elapsed: ~{len(subset_pdfs) * INTER_REQUEST_DELAY:.0f}s")
print(f"Watch tqdm: each iteration should take ~{INTER_REQUEST_DELAY}s")
print("-" * 60)


subset_results = []
subset_failed  = []
for entry in tqdm(subset_pdfs, desc="Subset verify", unit="pdf"):
    pdf_path = entry['path']
    iter_start = time.time()

    try:
        ocr_text, _, _ = run_typhoon_ocr_full_page(pdf_path)
        votes = parse_vote_counts(ocr_text)
        subset_results.append({'metadata': entry, 'votes': votes})
        elapsed = time.time() - iter_start
        tqdm.write(f"  OK  {entry['station']:40s}  api={elapsed:.1f}s")
    except Exception as exc:
        subset_failed.append({'pdf_path': pdf_path, 'error_reason': str(exc)})
        elapsed = time.time() - iter_start
        tqdm.write(f"  ERR {entry['station']:40s}  {exc}")
    finally:
        time.sleep(INTER_REQUEST_DELAY)  # rate limiting enforced here too

print("\nSubset verification done.")
print(f"  Success: {len(subset_results)} / Failed: {len(subset_failed)}")

# Show first successful extraction's parsed votes
if subset_results:
    print("\nFirst successful extraction votes:")
    for k, v in subset_results[0]['votes'].items():
        print(f"  {k}: {v}")

Running subset verification on 5 PDFs
Expected elapsed: ~15s
Watch tqdm: each iteration should take ~3.0s
------------------------------------------------------------


Subset verify:   0%|          | 0/5 [00:00<?, ?pdf/s]

  OK  หน่วยเลือกตั้งที่ 1                       api=4.1s
  OK  หน่วยเลือกตั้งที่ 10                      api=5.0s
  OK  หน่วยเลือกตั้งที่ 11                      api=4.8s
  ERR หน่วยเลือกตั้งที่ 12                      Typhoon OCR timed out after 20s
  OK  หน่วยเลือกตั้งที่ 13                      api=4.5s

Subset verification done.
  Success: 4 / Failed: 1

First successful extraction votes:
  eligible_voters: 423
  ballots_issued: None
  valid_ballots: 390
  spoiled_ballots: 18
  no_vote_ballots: None
  total_votes: 390
  candidate_votes: {}


## Cell 6: Main Extraction Loop (Throttled, Error-Logged)

Implements D-02 (error logging) and D-03 (rate limiting).

- `time.sleep(3)` between API calls → max 20 req/min (well within limit)
- Try-except captures any failure per file and logs it without crashing
- `failed_extractions` list accumulates failures for re-run

In [6]:
if not TYPHOON_API_KEY:
    print("ERROR: TYPHOON_OCR_API_KEY not set.")
    raise EnvironmentError("TYPHOON_OCR_API_KEY is required.")

extraction_results: list[dict] = []
failed_extractions: list[dict] = []
run_started_at = datetime.now(UTC).isoformat()

total = len(all_pdfs)
print(f"Starting batch extraction of {total} PDFs")
print(f"Rate limit: 1 request every {INTER_REQUEST_DELAY}s  (max {MAX_REQUESTS_PER_MIN}/min)")
print(f"Estimated time: ~{total * INTER_REQUEST_DELAY / 60:.1f} minutes")
print("-" * 70)

for idx, entry in enumerate(tqdm(all_pdfs, desc="Typhoon OCR", unit="pdf"), start=1):
    pdf_path = entry['path']
    iter_start = time.time()

    try:
        ocr_text, best_page, grid_score = run_typhoon_ocr_full_page(pdf_path)
        votes = parse_vote_counts(ocr_text)
        extraction_results.append({
            'pdf_path':    pdf_path,
            'district':    entry['district'],
            'subdistrict': entry['subdistrict'],
            'station':     entry['station'],
            'raw_text':    ocr_text,
            **votes,
        })
        elapsed = time.time() - iter_start
        tqdm.write(
            f"  [{idx:>3}/{total}] OK    {entry['filename']:<45s}"
            f"  p{best_page+1}  score={grid_score:.0f}  api={elapsed:.1f}s"
            f"  ok={len(extraction_results)}  fail={len(failed_extractions)}"
        )

    except Exception as exc:
        elapsed = time.time() - iter_start
        error_record = {
            'pdf_path':     pdf_path,
            'district':     entry.get('district', ''),
            'subdistrict':  entry.get('subdistrict', ''),
            'station':      entry.get('station', ''),
            'error_reason': str(exc),
            'error_type':   type(exc).__name__,
            'timestamp':    datetime.now(UTC).isoformat(),
        }
        failed_extractions.append(error_record)
        tqdm.write(
            f"  [{idx:>3}/{total}] FAIL  {entry['filename']:<45s}"
            f"  api={elapsed:.1f}s  {type(exc).__name__}: {exc}"
        )

    finally:
        time.sleep(INTER_REQUEST_DELAY)

print("\n" + "=" * 70)
print(f"Extraction complete")
print(f"  Successful: {len(extraction_results)}")
print(f"  Failed:     {len(failed_extractions)}")
print(f"  Total:      {total}")


Starting batch extraction of 133 PDFs
Rate limit: 1 request every 3.0s  (max 20/min)
Estimated time: ~6.7 minutes
----------------------------------------------------------------------


Typhoon OCR:   0%|          | 0/133 [00:00<?, ?pdf/s]

  [  1/133] OK    ส.ส.5ทับ18 1.pdf                               api=4.6s  ok=1  fail=0
  [  2/133] OK    ส.ส.5ทับ18 10.pdf                              api=5.1s  ok=2  fail=0
  [  3/133] OK    ส.ส.5ทับ18  11.pdf                             api=5.5s  ok=3  fail=0
  [  4/133] FAIL  ส.ส.5ทับ18 12.pdf                              api=20.1s  TimeoutError: Typhoon OCR timed out after 20s
  [  5/133] OK    ส.ส.5ทับ18  13.pdf                             api=6.8s  ok=4  fail=1
  [  6/133] OK    ส.ส.5ทับ18  14.pdf                             api=4.2s  ok=5  fail=1
  [  7/133] OK    ส.ส.5ทับ18  15.pdf                             api=6.8s  ok=6  fail=1
  [  8/133] OK    ส.ส.5ทับ18 2.pdf                               api=4.2s  ok=7  fail=1
  [  9/133] FAIL  ส.ส.5ทับ18 3.pdf                               api=20.1s  TimeoutError: Typhoon OCR timed out after 20s
  [ 10/133] OK    ส.ส.5ทับ18 4.pdf                               api=7.1s  ok=8  fail=2
  [ 11/133] FAIL  ส.ส.5ทับ18 5.pdf                  

## Cell 7: Write Failed Extractions Log (D-02)

In [7]:
# ---------------------------------------------------------------------------
# Write failed_extractions.json
# Structure designed for re-run: each entry has pdf_path + error_reason
# ---------------------------------------------------------------------------
failed_log_path = LOGS_DIR / "failed_extractions.json"

log_payload = {
    'run_started_at':  run_started_at,
    'run_finished_at': datetime.now(UTC).isoformat(),
    'total_processed': len(all_pdfs),
    'total_failed':    len(failed_extractions),
    'failed_extractions': failed_extractions,
}

with open(failed_log_path, 'w', encoding='utf-8') as f:
    json.dump(log_payload, f, ensure_ascii=False, indent=2)

print(f"Failed extractions log written to: {failed_log_path}")
print(f"  Total failed: {len(failed_extractions)}")
if failed_extractions:
    print("  Failed paths:")
    for rec in failed_extractions:
        print(f"    {rec['pdf_path']} — {rec['error_reason']}")

Failed extractions log written to: /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/logs/failed_extractions.json
  Total failed: 9
  Failed paths:
    /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2/อำเภอบ้านไร่/ตำบลทัพหลวง/หน่วยเลือกตั้งที่ 12/ส.ส.5ทับ18 12.pdf — Typhoon OCR timed out after 20s
    /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2/อำเภอบ้านไร่/ตำบลทัพหลวง/หน่วยเลือกตั้งที่ 3/ส.ส.5ทับ18 3.pdf — Typhoon OCR timed out after 20s
    /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2/อำเภอบ้านไร่/ตำบลทัพหลวง/หน่วยเลือกตั้งที่ 5/ส.ส.5ทับ18 5.pdf — Typhoon OCR timed out after 20s
    /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2/อำเภอบ้านไร่/ตำบลทัพหลวง/หน่วยเลือกตั้งที่ 7/ส.ส. 5ทับ18  7.pdf — Typhoon OCR timed out after 20s
    /home/chatrin/Documents/Chat/

## Cell 9: Error Injection Test

Verify D-02 error handling: temporarily use a bad API key to trigger failures,
confirm the loop continues and writes to `logs/failed_extractions.json`.

In [9]:
# ---------------------------------------------------------------------------
# Error injection test — uses a deliberately bad path to simulate a failure.
# This tests that the error handling works WITHOUT making live API calls.
# ---------------------------------------------------------------------------
print("=== Error injection test ===")

test_entries = [
    {'path': '/nonexistent/bad_path_1.pdf', 'district': 'test', 'subdistrict': 'test', 'station': 'bad_1'},
    {'path': '/nonexistent/bad_path_2.pdf', 'district': 'test', 'subdistrict': 'test', 'station': 'bad_2'},
]

test_failed: list[dict] = []
test_succeeded: list[dict] = []

for entry in test_entries:
    pdf_path = entry['path']
    try:
        ocr_text, _, _ = run_typhoon_ocr_full_page(pdf_path)
        test_succeeded.append({'pdf_path': pdf_path})
    except Exception as exc:
        # D-02: catch, log, continue
        error_record = {
            'pdf_path':    pdf_path,
            'error_reason': str(exc),
            'error_type':   type(exc).__name__,
            'timestamp':    datetime.now(UTC).isoformat(),
        }
        test_failed.append(error_record)
        print(f"  Caught error (expected): {exc}")
    finally:
        time.sleep(0.1)  # minimal delay for test

print(f"Loop completed without crash.")
print(f"  test_succeeded: {len(test_succeeded)}")
print(f"  test_failed:    {len(test_failed)}")

# Write test-mode failure log to verify the JSON serialisation works
test_log_path = LOGS_DIR / "failed_extractions_test.json"
test_log_payload = {
    'mode': 'error_injection_test',
    'run_at': datetime.now(UTC).isoformat(),
    'total_failed': len(test_failed),
    'failed_extractions': test_failed,
}
with open(test_log_path, 'w', encoding='utf-8') as f:
    json.dump(test_log_payload, f, ensure_ascii=False, indent=2)

print(f"\nTest failure log written: {test_log_path}")
print("D-02 error handling verified: loop continues, failures logged.")

=== Error injection test ===
  Caught error (expected): no such file: '/nonexistent/bad_path_1.pdf'
  Caught error (expected): no such file: '/nonexistent/bad_path_2.pdf'
Loop completed without crash.
  test_succeeded: 0
  test_failed:    2

Test failure log written: /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/logs/failed_extractions_test.json
D-02 error handling verified: loop continues, failures logged.


---
## Cell 10: Retry Mechanism — Reprocess `logs/failed_extractions.json` (Plan 02-02, Task 1)

Reads the structured error log produced by Cell 6 and re-invokes the Typhoon OCR pipeline
**only** for previously failed PDFs. Successful retries are appended to `extraction_results`
and removed from `failed_extractions`.

> **Run this cell AFTER the main extraction loop (Cell 6) has completed.**

In [10]:
# ---------------------------------------------------------------------------
# Retry mechanism: re-run OCR only on entries from logs/failed_extractions.json
# ---------------------------------------------------------------------------
failed_log_path = LOGS_DIR / "failed_extractions.json"

retry_results: list[dict] = []     # successful retries this pass
still_failed: list[dict] = []      # entries that failed again on retry

if not failed_log_path.exists():
    print(f"No failed log found at {failed_log_path}. Nothing to retry.")
else:
    with open(failed_log_path, encoding='utf-8') as fh:
        log_data = json.load(fh)

    retry_candidates = log_data.get('failed_extractions', [])
    print(f"Found {len(retry_candidates)} previously failed extractions to retry.")

    if not retry_candidates:
        print("  All extractions succeeded on the first pass — nothing to retry.")
    else:
        print(f"  Retrying {len(retry_candidates)} PDFs...")
        print(f"  Rate limit: 1 request every {INTER_REQUEST_DELAY}s")
        print("-" * 60)

        for rec in tqdm(retry_candidates, desc="Retry OCR", unit="pdf"):
            pdf_path = rec['pdf_path']
            district = rec.get('district', '')
            subdistrict = rec.get('subdistrict', '')
            station = rec.get('station', '')

            try:
                ocr_text, _, _ = run_typhoon_ocr_full_page(pdf_path)
                votes = parse_vote_counts(ocr_text)

                success_record = {
                    'pdf_path':    pdf_path,
                    'district':    district,
                    'subdistrict': subdistrict,
                    'station':     station,
                    'raw_text':    ocr_text,
                    'retry_pass':  2,
                    **votes,
                }
                retry_results.append(success_record)
                # Append to master results list so it flows into the dataframe
                extraction_results.append(success_record)
                tqdm.write(f"  RETRY OK: {pdf_path}")

            except Exception as exc:
                updated_rec = {
                    **rec,
                    'retry_error_reason': str(exc),
                    'retry_error_type': type(exc).__name__,
                    'retry_timestamp': datetime.now(UTC).isoformat(),
                }
                still_failed.append(updated_rec)
                tqdm.write(f"  STILL FAILED: {pdf_path} — {exc}")

            finally:
                # Respect rate limits on retry pass too
                time.sleep(INTER_REQUEST_DELAY)

        # Update the failed_extractions list in-memory (remove recovered entries)
        failed_extractions.clear()
        failed_extractions.extend(still_failed)

        # Persist updated failure log (only entries that failed both passes)
        updated_log = {
            'run_started_at':    log_data.get('run_started_at', ''),
            'run_finished_at':   datetime.now(UTC).isoformat(),
            'total_processed':   log_data.get('total_processed', 0),
            'retry_pass2_count': len(retry_candidates),
            'retry_recovered':   len(retry_results),
            'total_failed':      len(still_failed),
            'failed_extractions': still_failed,
        }
        with open(failed_log_path, 'w', encoding='utf-8') as fh:
            json.dump(updated_log, fh, ensure_ascii=False, indent=2)

        print("\n" + "=" * 60)
        print(f"Retry complete")
        print(f"  Recovered:    {len(retry_results)}")
        print(f"  Still failed: {len(still_failed)}")
        print(f"  Master total: {len(extraction_results)} successful extractions")
        print(f"  Log updated:  {failed_log_path}")


Found 9 previously failed extractions to retry.
  Retrying 9 PDFs...
  Rate limit: 1 request every 3.0s
------------------------------------------------------------


Retry OCR:   0%|          | 0/9 [00:00<?, ?pdf/s]

  STILL FAILED: /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2/อำเภอบ้านไร่/ตำบลทัพหลวง/หน่วยเลือกตั้งที่ 12/ส.ส.5ทับ18 12.pdf — Typhoon OCR timed out after 20s
  STILL FAILED: /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2/อำเภอบ้านไร่/ตำบลทัพหลวง/หน่วยเลือกตั้งที่ 3/ส.ส.5ทับ18 3.pdf — Typhoon OCR timed out after 20s
  RETRY OK: /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2/อำเภอบ้านไร่/ตำบลทัพหลวง/หน่วยเลือกตั้งที่ 5/ส.ส.5ทับ18 5.pdf
  STILL FAILED: /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2/อำเภอบ้านไร่/ตำบลทัพหลวง/หน่วยเลือกตั้งที่ 7/ส.ส. 5ทับ18  7.pdf — Typhoon OCR timed out after 20s
  STILL FAILED: /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2/อำเภอบ้านไร่/ตำบลบ้านบึง/หน่วยเลือกตั้งที่ 2/ส.ส.5ทับ18.pdf — Typhoon OCR timed 

---
## Cell 11: Validation Logic — Vote Sum Integrity Check (Plan 02-02, Task 2)

Iterates over `extraction_results` and checks whether `sum(candidate_votes)` matches
`total_votes` parsed from the document. Mismatches are collected in `validation_mismatches`
for manual review.

Per D-04: **No mismatch flag is written to the CSV.** The flag is a notebook-only variable.

Mismatching rows are still included in the final export dataframe — they are NOT dropped.

In [11]:
import pandas as pd

# ---------------------------------------------------------------------------
# Build master dataframe from all successful extractions
# ---------------------------------------------------------------------------
df = pd.DataFrame(extraction_results)
print(f"Master dataframe shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# ---------------------------------------------------------------------------
# DATA-02: Vote sum validation
# Track mismatches in a list — do NOT add a flag column to the dataframe
# ---------------------------------------------------------------------------
validation_mismatches: list[dict] = []

for idx, row in df.iterrows():
    parsed_total = row.get('total_votes')  # the 'รวมคะแนน' field from OCR
    candidate_votes_dict = row.get('candidate_votes', {})

    if parsed_total is None or not isinstance(candidate_votes_dict, dict):
        # Cannot validate if either value is missing — skip silently
        continue

    computed_total = sum(candidate_votes_dict.values())

    if computed_total != parsed_total:
        validation_mismatches.append({
            'row_index':    idx,
            'pdf_path':     row.get('pdf_path', ''),
            'station':      row.get('station', ''),
            'parsed_total': parsed_total,
            'computed_sum': computed_total,
            'difference':   computed_total - parsed_total,
        })

# ---------------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------------
print(f"\nValidation summary:")
print(f"  Rows evaluated:  {len(df)}")
print(f"  Mismatches found: {len(validation_mismatches)}")

if validation_mismatches:
    print("\n  *** MISMATCHES — inspect these stations manually: ***")
    mismatch_df = pd.DataFrame(validation_mismatches)
    display(mismatch_df)
else:
    print("  All vote sums match their parsed totals.")

# Confirm no mismatch column exists in df
assert 'mismatch' not in [c.lower() for c in df.columns], \
    "Mismatch flag must NOT be stored in the dataframe (D-04)"
print("\n[OK] No mismatch flag column in dataframe — D-04 satisfied.")


Master dataframe shape: (127, 13)
Columns: ['pdf_path', 'district', 'subdistrict', 'station', 'raw_text', 'eligible_voters', 'ballots_issued', 'valid_ballots', 'spoiled_ballots', 'no_vote_ballots', 'total_votes', 'candidate_votes', 'retry_pass']

Validation summary:
  Rows evaluated:  127
  Mismatches found: 127

  *** MISMATCHES — inspect these stations manually: ***


,row_index,pdf_path,station,parsed_total,computed_sum,difference
0,0,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 1,390.0,0,-390.0
1,1,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 10,NaN,0,NaN
2,2,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 11,207.0,0,-207.0
3,3,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 13,176.0,0,-176.0
4,4,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 14,237.0,0,-237.0
...,...,...,...,...,...,...
122,122,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 10,214.0,0,-214.0
123,123,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 11,256.0,0,-256.0
124,124,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 5,321.0,0,-321.0
125,125,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 6,NaN,0,NaN



[OK] No mismatch flag column in dataframe — D-04 satisfied.


---
## Cell 12: Data Cleanup & Export — `output/election_structured_data.csv` (Plan 02-02, Task 3)

Applies OCR numerical mis-interpretation corrections before export:
- Letter `O` (uppercase oh) -> digit `0` in numeric string columns
- Letter `l` (lowercase ell) -> digit `1` in numeric string columns
- Letter `I` (uppercase eye) -> digit `1` in numeric string columns
- Letter `S` (uppercase S) -> digit `5` (common OCR error)
- Letter `B` (uppercase B) -> digit `8` (common OCR error)

Saves the cleaned dataframe to `output/election_structured_data.csv`.

All rows (including mismatches) are exported — no rows are dropped.

In [12]:
# ---------------------------------------------------------------------------
# Output directory
# ---------------------------------------------------------------------------
OUTPUT_DIR = NOTEBOOK_DIR.parent / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / 'election_structured_data.csv'

# ---------------------------------------------------------------------------
# DATA-03: Clean common OCR numerical misinterpretations
# Applies only to columns whose intended values are numeric strings.
# The raw_text column is intentionally skipped to preserve original OCR output.
# ---------------------------------------------------------------------------
NUMERIC_STRING_COLS = [
    'eligible_voters', 'ballots_issued', 'valid_ballots',
    'spoiled_ballots', 'no_vote_ballots', 'total_votes',
]

# OCR mis-read character -> correct character mapping for numeric fields
OCR_NUMERIC_FIXES = {
    'O': '0',   # uppercase Oh  -> zero
    'l': '1',   # lowercase ell -> one
    'I': '1',   # uppercase eye -> one
    'S': '5',   # uppercase S   -> five (common OCR error)
    'B': '8',   # uppercase B   -> eight (common OCR error)
}

df_clean = df.copy()

# Apply fixes to each numeric string column
for col in NUMERIC_STRING_COLS:
    if col in df_clean.columns:
        # Convert to string, apply replacements, attempt numeric coercion
        col_str = df_clean[col].astype(str)
        for wrong_char, correct_char in OCR_NUMERIC_FIXES.items():
            col_str = col_str.str.replace(wrong_char, correct_char, regex=False)
        # Re-cast to numeric (NaN if still un-parseable after fixes)
        df_clean[col] = pd.to_numeric(col_str, errors='coerce')

# ---------------------------------------------------------------------------
# Drop internal/debug columns not needed in the final export
# ---------------------------------------------------------------------------
EXPORT_COLS_EXCLUDE = ['raw_text', 'retry_pass']
export_cols = [c for c in df_clean.columns if c not in EXPORT_COLS_EXCLUDE]

df_export = df_clean[export_cols].copy()

# Expand candidate_votes dict into individual columns (cand_1_votes, cand_2_votes, ...)
if 'candidate_votes' in df_export.columns:
    # Expand dict column into separate per-candidate columns
    cand_df = df_export['candidate_votes'].apply(
        lambda d: {f'cand_{k}_votes': v for k, v in d.items()} if isinstance(d, dict) else {}
    )
    cand_expanded = pd.DataFrame(list(cand_df), index=df_export.index)
    df_export = df_export.drop(columns=['candidate_votes'])
    df_export = pd.concat([df_export, cand_expanded], axis=1)

# ---------------------------------------------------------------------------
# Confirm D-04: no mismatch flag in the exported dataframe
# ---------------------------------------------------------------------------
assert 'mismatch' not in [c.lower() for c in df_export.columns], \
    "Mismatch flag must NOT be present in the exported CSV (D-04)"

# ---------------------------------------------------------------------------
# Export to CSV
# ---------------------------------------------------------------------------
df_export.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print(f"Export complete: {OUTPUT_PATH}")
print(f"  Rows exported: {len(df_export)}")
print(f"  Columns:       {list(df_export.columns)}")
print()

# ---------------------------------------------------------------------------
# Final validation_mismatches display for human review
# ---------------------------------------------------------------------------
print(f"validation_mismatches: {len(validation_mismatches)} row(s) flagged for manual review")
if validation_mismatches:
    display(pd.DataFrame(validation_mismatches))
else:
    print("  No mismatches detected — all extracted vote totals are internally consistent.")

print("\n[OK] election_structured_data.csv written — DATA-03 satisfied.")
print("[OK] No mismatch flag in CSV — D-04 satisfied.")


Export complete: /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/output/election_structured_data.csv
  Rows exported: 127
  Columns:       ['pdf_path', 'district', 'subdistrict', 'station', 'eligible_voters', 'ballots_issued', 'valid_ballots', 'spoiled_ballots', 'no_vote_ballots', 'total_votes']

validation_mismatches: 127 row(s) flagged for manual review


,row_index,pdf_path,station,parsed_total,computed_sum,difference
0,0,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 1,390.0,0,-390.0
1,1,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 10,NaN,0,NaN
2,2,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 11,207.0,0,-207.0
3,3,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 13,176.0,0,-176.0
4,4,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 14,237.0,0,-237.0
...,...,...,...,...,...,...
122,122,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 10,214.0,0,-214.0
123,123,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 11,256.0,0,-256.0
124,124,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 5,321.0,0,-321.0
125,125,/home/chatrin/Documents/Chat/CU/Year-3/2110446...,หน่วยเลือกตั้งที่ 6,NaN,0,NaN



[OK] election_structured_data.csv written — DATA-03 satisfied.
[OK] No mismatch flag in CSV — D-04 satisfied.
